In [10]:
from dotenv import load_dotenv
load_dotenv()

True

# Parser

In [11]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./input.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

107


# RAG Builder

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 100,
    chunk_overlap= 10,
    add_start_index= True,
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

4938


In [13]:
all_splits_text = []
for i, split in enumerate(all_splits):
    all_splits_text.append(split.page_content)

In [14]:
import chromadb

collection = chromadb.PersistentClient(path="./chroma_db").create_collection(
    name= "Nike-10K-Form",
    metadata= {"description": "Nike 10K Form"},
    get_or_create= True,
)

In [15]:
collection.add(
    documents= all_splits_text,
    ids= [f"doc_{i}" for i in range(len(all_splits_text))],
)

In [27]:
from typing import Annotated
from pydantic import Field

class RAGAgent:
    def __init__(self, collection):
        self.collection = collection

    def get_rag_context(query: Annotated[str, Field(description="The query to retrieve context from the database")]) -> str:
        """Queries a RAG database to provide context on the company Nike"""
    
        results = collection.query(
            query_texts= query,
            include= ["documents", "metadatas"],
            n_results= 2
        )

        context_entries = []
        if results and results.get("documents") and results["documents"][0]:
            for doc, meta, in zip(results["documents"][0], results["metadatas"][0]):
                context_entries.append(f"Document: {doc}\nMetadata: {meta}")
        result = "\n\n".join(context_entries) if context_entries else "No retrieval context found"

        print(f"result: {result}")
        return result

rag_agent = RAGAgent(collection)

In [28]:
from agent_framework.openai import OpenAIChatClient
from os import environ

client = OpenAIChatClient(
    base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
    api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
    model_id= environ.get("GITHUB_MODEL_ID")  # 🎯 Selected AI model
)

nike_agent = client.as_agent(
    name= "NikeAgent",
    instructions= """You are a helpful agent to answer queries about the company Nike.
    Answer only using the provided context from the RAG database.""",
    tools= [rag_agent.get_rag_context]
)

In [29]:
from agent_framework import WorkflowBuilder

workflow = WorkflowBuilder(start_executor=nike_agent).build()

In [30]:
result = await workflow.run("What is the main type of business Nike Performs?")

In [31]:
for output in result.get_outputs():
    output = str.replace(output.text, ". ", ".\n")
    print(f"{output}\n")

I'm currently unable to pull information from the database.
However, I can tell you that Nike primarily operates in the athletic footwear and apparel industry, designing, manufacturing, and marketing a wide range of sports-related products.
If you have any other questions or need information on a specific aspect of Nike, feel free to ask!

